# Day 3 - Part 3: Seq2Seq와 어텐션 - 실습 과제

### 목표: 영어-한글 번역기 만들기

이번 과제에서는 튜토리얼에서 배운 Seq2Seq와 어텐션 모델을 활용하여, 간단한 영어 문장을 한글 문장으로 번역하는 모델을 직접 구축하고 학습시켜 봅니다. 

튜토리얼의 '문장 뒤집기' 예제가 모델 구조 자체에 대한 이해를 도왔다면, 이번 과제는 실제 번역 문제에 Seq2Seq를 적용하는 실전 경험을 제공합니다.

**과제 수행 절차:**
1. `데이터 준비:` 제공된 코드를 통해 영어-한국어 병렬 코퍼스를 로드하고, 각 언어에 맞는 어휘 사전을 구축하여 데이터를 전처리합니다.

2. `모델 완성:` 튜토리얼에서 사용된 `Encoder`, `LuongAttention`, `Decoder` 코드의 일부 빈칸(`# TODO`)을 채워 모델을 완성합니다.
3. `학습 루프 완성:` 모델 학습을 위한 `train` 함수의 일부 빈칸(`# TODO`)을 채워 학습 과정을 구현합니다.
4. `모델 학습 및 평가:` 완성된 모델과 학습 코드를 실행하여 번역 모델을 학습시키고, 테스트 데이터에 대한 손실(Loss)을 측정하여 성능을 평가합니다.
5. `번역기 테스트:` 직접 만든 `translate_sentence` 함수를 통해 새로운 영어 문장을 한국어로 번역해보고, 어텐션 맵을 시각화하여 번역 과정을 분석합니다.

각 단계별로 `# TODO:` 주석이 달린 부분을 채워주세요.

### 1. 라이브러리 임포트 및 기본 설정

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import re
import random
from collections import Counter
from tqdm import tqdm
import plotly.express as px

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('mps') # mac
print(f'Using device: {device}')

# 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Using device: mps


### 2. 데이터 준비 (영어-한글 병렬 코퍼스)
Tatoeba 프로젝트에서 제공하는 영어-한글 번역 데이터셋을 사용합니다. 데이터셋은 `영어 문장<TAB>한글 문장` 형태로 구성되어 있습니다.

In [10]:
# 데이터셋 다운로드 및 압축 해제
!mkdir -p ../../datasets/dl/corpus/
!wget http://www.manythings.org/anki/kor-eng.zip -O ../../datasets/dl/corpus/kor-eng.zip
!unzip -o ../../datasets/dl/corpus/kor-eng.zip -d ../../datasets/dl/corpus/
!rm ../../datasets/dl/corpus/kor-eng.zip

--2025-06-25 09:30:01--  http://www.manythings.org/anki/kor-eng.zip
www.manythings.org (www.manythings.org) 해석 중... 173.254.30.110
다음으로 연결 중: www.manythings.org (www.manythings.org)|173.254.30.110|:80... 연결했습니다.
HTTP 요청을 보냈습니다. 응답 기다리는 중... 200 OK
길이: 246721 (241K) [application/zip]
저장 위치: `../../datasets/dl/corpus/kor-eng.zip'

../../datasets/dl/c 100%[===================>] 240.94K   395KB/s    /  0.6s    

2025-06-25 09:30:02 (395 KB/s) - `../../datasets/dl/corpus/kor-eng.zip' 저장함 [246721/246721]

Archive:  ../../datasets/dl/corpus/kor-eng.zip
  inflating: ../../datasets/dl/corpus//_about.txt  
  inflating: ../../datasets/dl/corpus//kor.txt  


In [11]:
DATA_PATH = '../../datasets/dl/corpus/kor.txt'

In [26]:
def unicode_to_ascii(s):
    return ''.join(
        c for c in s if c in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ' or c in " .!?,'"
    )

def preprocess_sentence(w):
    w = w.lower().strip()
    w = unicode_to_ascii(w)
    w = re.sub(r"([?.!,'])", r" \1 ", w)
    w = re.sub(r'[ ]+', ' ', w).strip()
    return w

def preprocess_korean(w):
    w = w.strip()
    # 한국어 문장 부호 주변에 공백 추가
    w = re.sub(r"([.!?])", r" \1 ", w)
    w = re.sub(r'[ ]+', ' ', w).strip()
    return w

def create_dataset(path, num_examples=30000):
    lines = open(path, encoding='UTF-8').read().strip().split('\n')
    word_pairs = []
    
    for l in lines[:num_examples]:
        parts = l.split('\t')
        if len(parts) >= 2:
            english = preprocess_sentence(parts[0])
            korean = preprocess_korean(parts[1])
            word_pairs.append([english, korean])
    
    return zip(*word_pairs)

en, ko = create_dataset(DATA_PATH)
print("Sample English sentence:", en[10])
print("Sample Korean sentence:", ko[10])
print("Total pairs:", len(en))

Sample English sentence: jump !
Sample Korean sentence: 점프 !
Total pairs: 6245


#### 어휘 사전(Vocabulary) 구축
소스 언어(영어)와 타겟 언어(한국어)에 대해 각각 별도의 어휘 사전을 만듭니다.

In [28]:
class Vocab:
    def __init__(self, lang_sents):
        self.pad_token, self.sos_token, self.eos_token, self.unk_token = 0, 1, 2, 3
        self.word2idx = {'<pad>': self.pad_token, '<sos>': self.sos_token, '<eos>': self.eos_token, '<unk>': self.unk_token}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.word_count = {}
        
        for sent in lang_sents:
            for word in sent.split(' '):
                if word not in self.word2idx:
                    new_idx = len(self.word2idx)
                    self.word2idx[word] = new_idx
                    self.idx2word[new_idx] = word
                self.word_count[word] = self.word_count.get(word, 0) + 1
    
    def __len__(self):
        return len(self.word2idx)

input_lang = Vocab(en)
output_lang = Vocab(ko)

print("Input (영어) Vocab size:", len(input_lang))
print("Output (한국어) Vocab size:", len(output_lang))

Input (영어) Vocab size: 3251
Output (한국어) Vocab size: 8430


#### 데이터셋 및 데이터로더 생성
PyTorch의 `Dataset`과 `DataLoader`를 사용하여 모델에 데이터를 공급할 준비를 합니다.

In [29]:
class TranslationDataset(Dataset):
    def __init__(self, input_sents, output_sents, input_vocab, output_vocab):
        self.input_sents = input_sents
        self.output_sents = output_sents
        self.input_vocab = input_vocab
        self.output_vocab = output_vocab

    def __len__(self):
        return len(self.input_sents)

    def __getitem__(self, idx):
        input_tokens = [self.input_vocab.word2idx.get(w, self.input_vocab.unk_token) for w in self.input_sents[idx].split(' ')] + [self.input_vocab.eos_token]
        output_tokens = [self.output_vocab.sos_token] + [self.output_vocab.word2idx.get(w, self.output_vocab.unk_token) for w in self.output_sents[idx].split(' ')] + [self.output_vocab.eos_token]
        return torch.tensor(input_tokens), torch.tensor(output_tokens)

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    
    src_lens = [len(s) for s in srcs]
    tgt_lens = [len(t) for t in tgts]

    # pad_sequence를 사용하여 패딩 처리
    padded_srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=input_lang.pad_token)
    padded_tgts = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=output_lang.pad_token)
    
    return padded_srcs, torch.tensor(src_lens), padded_tgts

# 데이터셋 및 데이터로더 인스턴스화
train_dataset = TranslationDataset(en, ko, input_lang, output_lang)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

### 3. Seq2Seq 모델 완성하기

튜토리얼에서 배운 내용을 바탕으로 아래 모델 코드의 `# TODO` 부분을 완성하여 전체 번역 모델을 구축하세요.

In [30]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=input_lang.pad_token)
        self.rnn = nn.LSTM(emb_dim, hid_dim, 
                           num_layers=n_layers, dropout=dropout, 
                           bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src, src_len):
        embedded = self.dropout(self.embedding(src))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_len.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, (h, c) = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        h_cat = torch.cat((h[-2,:,:], h[-1,:,:]), dim=1)
        c_cat = torch.cat((c[-2,:,:], c[-1,:,:]), dim=1)
        hidden = torch.tanh(self.fc(h_cat)).unsqueeze(0)
        cell = torch.tanh(self.fc(c_cat)).unsqueeze(0)
        return outputs, hidden, cell

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 3, hid_dim) 
        self.v = nn.Linear(hid_dim, 1, bias = False)
    
    def forward(self, hidden, encoder_outputs, mask):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1) 
        
        # 과제: 어텐션 에너지(스코어)를 계산하세요.
        # HINT: hidden과 encoder_outputs를 concat하고, self.attn 레이어를 통과시킨 후 tanh 활성화 함수를 적용합니다.
        energy = # TODO: 여기에 코드 작성

        attention = self.v(energy).squeeze(2)
        attention = attention.masked_fill(mask == 0, -1e10)
        attn_weights = torch.softmax(attention, dim=1) 
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        return context, attn_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, attn, n_layers=2, dropout=0.2):
        super().__init__()
        self.output_dim = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=output_lang.pad_token)
        self.rnn = nn.LSTM(emb_dim + hid_dim * 2, hid_dim, num_layers=n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.attn = attn
        
        # 과제: 최종 단어 예측을 위한 FC 레이어를 정의하세요.
        # HINT: 입력 크기는 LSTM 출력(hid_dim), 컨텍스트 벡터(hid_dim*2), 임베딩(emb_dim)의 합입니다.
        self.fc_out = # TODO: 여기에 코드 작성

    def forward(self, input_tok, hidden, cell, encoder_outputs, mask):
        input_tok = input_tok.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_tok))
        context, attn_weights = self.attn(hidden.squeeze(0), encoder_outputs, mask)
        rnn_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        pred = self.fc_out(torch.cat((output.squeeze(1), context.squeeze(1), embedded.squeeze(1)), dim=1))
        return pred, hidden, cell, attn_weights

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def create_mask(self, src):
        return (src != input_lang.pad_token)

    def forward(self, src, src_len, tgt, teacher_forcing_ratio=0.5):
        B, max_len_tgt = tgt.shape
        vocab_size = self.decoder.output_dim
        outputs = torch.zeros(B, max_len_tgt, vocab_size).to(device)
        enc_out, hidden, cell = self.encoder(src, src_len)
        mask = self.create_mask(src)
        input_tok = tgt[:, 0]
        for t in range(1, max_len_tgt):
            pred, hidden, cell, _ = self.decoder(input_tok, hidden, cell, enc_out, mask)
            outputs[:, t] = pred
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = pred.argmax(1)
            input_tok = tgt[:, t] if teacher_force else top1
        return outputs

### 4. 모델 학습 및 평가
모델 인스턴스를 생성하고, 학습 루프를 완성하여 번역 모델 학습을 시작합니다.

In [ ]:
# 하이퍼파라미터
INPUT_DIM = len(input_lang)
OUTPUT_DIM = len(output_lang)
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HID_DIM = 256
N_LAYERS = 2
ENC_DROPOUT = 0.3
DEC_DROPOUT = 0.3

# 모델 생성
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
attn = LuongAttention(HID_DIM)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, attn, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index = output_lang.pad_token)

# 학습 함수
def train(model, dataloader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for src, src_len, tgt in tqdm(dataloader, desc="Training"):
        src, tgt = src.to(device), tgt.to(device)
        
        # 과제: 모델 학습의 4단계를 구현하세요.
        # 1. 기울기 초기화
        # TODO: 여기에 코드 작성
        
        # 2. 모델 예측
        output = # TODO: 여기에 코드 작성
        
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        tgt = tgt[:, 1:].reshape(-1)
        
        # 3. 손실 계산
        loss = # TODO: 여기에 코드 작성
        
        # 4. 역전파 및 가중치 업데이트
        # TODO: 여기에 코드 작성 (3줄, 역전파, 기울기 클리핑, 옵티마이저 스텝 포함)
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(dataloader)

# 학습 시작
N_EPOCHS = 10
CLIP = 1

for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f}')

### 5. 번역기 테스트 및 어텐션 시각화
학습된 모델로 새로운 문장을 번역해보고, 어텐션이 어떻게 작동하는지 확인합니다.

In [ ]:
def translate_sentence(sentence, model, max_len=50):
    model.eval()
    
    # 전처리 및 토큰화
    processed_sent = preprocess_sentence(sentence)
    tokens = [t for t in processed_sent.split(' ')] + [input_lang.idx2word[input_lang.eos_token]]
    src_indexes = [input_lang.word2idx.get(t, input_lang.unk_token) for t in tokens]
    
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(device)
    src_len = torch.LongTensor([len(src_indexes)])
    
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor, src_len)
    
    mask = model.create_mask(src_tensor)
    tgt_indexes = [output_lang.sos_token]
    attentions = torch.zeros(max_len, 1, len(src_indexes)).to(device)

    for i in range(max_len):
        tgt_tensor = torch.LongTensor([tgt_indexes[-1]]).to(device)
        
        with torch.no_grad():
            output, hidden, cell, attention = model.decoder(tgt_tensor, hidden, cell, encoder_outputs, mask)
        
        attentions[i] = attention
        pred_token = output.argmax(1).item()
        tgt_indexes.append(pred_token)

        if pred_token == output_lang.eos_token:
            break
            
    tgt_tokens = [output_lang.idx2word[i] for i in tgt_indexes]
    return tgt_tokens[1:-1], tokens, attentions[:len(tgt_tokens)-1].squeeze(1).cpu()

# 번역 테스트
example_sentence = "i am a student ."
translation, original, attention = translate_sentence(example_sentence, model)

print('Original:', ' '.join(original))
print('Translated:', ' '.join(translation))

# 어텐션 시각화
fig = px.imshow(attention, x=original, y=translation, labels=dict(color="Attention"))
fig.update_xaxes(side="top")
fig.show()

### 6. 심화 과제

1. **GRU 사용**: `nn.LSTM`을 `nn.GRU`로 변경하여 모델을 다시 학습시켜 보세요. 학습 속도와 번역 성능에 어떤 변화가 있는지 비교해 보세요. (GRU는 cell 상태를 반환하지 않는 점에 유의하세요.)
   
2. **더 많은 데이터 사용**: `create_dataset` 함수의 `num_examples`를 늘려 더 많은 데이터로 학습시키고 성능 변화를 관찰해 보세요.
3. **하이퍼파라미터 튜닝**: `EMB_DIM`, `HID_DIM`, `N_LAYERS`, `DROPOUT` 등의 값을 변경하며 더 좋은 번역 성능을 내는 조합을 찾아보세요.
4. **빔 서치(Beam Search) 디코딩**: 현재 구현된 `greedy_decode` 방식(매번 가장 확률이 높은 단어만 선택) 대신, 여러 개의 후보를 유지하며 탐색하는 빔 서치 디코딩을 구현하여 번역 품질을 향상시켜 보세요.